In [13]:
import os
import re
import math
from collections import defaultdict

In [14]:
connector_words = {
    "a","an","the","they","these","this","for","is","are","was",
    "of","or","and","does","will","whose"
}

singular_map = {
    "stacks": "stack",
    "structures": "structure",
    "applications": "application"
}

In [15]:
def normalize_word(word):
    word = word.lower()
    word = re.sub(r"[{}\[\]<>=\(\)\.,;'\"?#!\-:]", "", word)

    if word in singular_map:
        word = singular_map[word]

    return word.strip()

In [16]:
print(normalize_word("Structures"))
print(normalize_word("applications"))
print(normalize_word("data,"))

structure
application
data


In [24]:
folder_path = r"Assignment 4- datasets/Q2- webSearch/webpages"

In [25]:
""" PageEntry Class - for each webpage it will 1. read file, 2. keep word position 3. store useful words"""

class PageEntry:
    def __init__(self, page_name, folder_path):
        self.page_name = page_name
        self.word_positions = defaultdict(list)
        self.total_words = 0
        
        filepath = os.path.join(folder_path, page_name)
        
        with open(filepath, "r", encoding="utf-8") as f:
            text = f.read()

        words = text.split()
        position = 0

        for w in words:
            position += 1
            clean = normalize_word(w)

            if clean == "":
                continue

            if clean not in connector_words:
                self.word_positions[clean].append(position)
                self.total_words += 1

In [26]:
"""InvertedPageIndex Class"""

class InvertedPageIndex:
    def __init__(self):
        self.index = defaultdict(list)
        self.pages = {}

    def addPage(self, page):
        self.pages[page.page_name] = page

        for word, positions in page.word_positions.items():
            for pos in positions:
                self.index[word].append((page.page_name, pos))

In [27]:
""" Search engine class"""

class SearchEngine:
    def __init__(self, folder_path):
        self.folder_path = folder_path
        self.db = InvertedPageIndex()

    def addPage(self, page_name):
        page = PageEntry(page_name, self.folder_path)
        self.db.addPage(page)

    def queryFindPagesWhichContainWord(self, word):
        word = normalize_word(word)

        if word not in self.db.index:
            print(f"No webpage contains word {word}")
            return

        pages = sorted(set(p for p, _ in self.db.index[word]))
        print(", ".join(pages))

    def queryFindPositionsOfWordInAPage(self, word, page_name):
        word = normalize_word(word)

        if page_name not in self.db.pages:
            print(f"No webpage {page_name} found")
            return

        page = self.db.pages[page_name]

        if word not in page.word_positions:
            print(f"Webpage {page_name} does not contain word {word}")
            return

        print(", ".join(map(str, page.word_positions[word])))

In [28]:
for file in os.listdir(folder_path):
    engine.addPage(file)

print("All pages added successfully")

All pages added successfully


In [31]:
"""testing queries"""

engine.queryFindPagesWhichContainWord("stack")
engine.queryFindPagesWhichContainWord("data")
engine.queryFindPagesWhichContainWord("structure")

"""testing positions"""

engine.queryFindPositionsOfWordInAPage("stack", "stack_cprogramming")
engine.queryFindPositionsOfWordInAPage("data", "stack_cprogramming")

stack_cprogramming, stack_datastructure_wiki, stack_oracle, stacklighting, stackmagazine, stackoverflow
stack_cprogramming, stack_datastructure_wiki, stackoverflow
stack_cprogramming, stack_datastructure_wiki
2, 13, 87, 109, 138, 156, 159, 180, 189, 197, 205, 223, 245
3, 17, 98, 113


In [32]:
actions_file = r"Assignment 4- datasets/Q2- webSearch/actions.txt"

with open(actions_file, "r", encoding="utf-8") as f:
    actions = f.readlines()

for action in actions:
    action = action.strip()

    if action.startswith("addPage"):
        _, page = action.split()
        engine.addPage(page)

    elif action.startswith("queryFindPagesWhichContainWord"):
        _, word = action.split()
        print("\n", action)
        engine.queryFindPagesWhichContainWord(word)

    elif action.startswith("queryFindPositionsOfWordInAPage"):
        parts = action.split()
        word = parts[1]
        page = parts[2]
        print("\n", action)
        engine.queryFindPositionsOfWordInAPage(word, page)


 queryFindPagesWhichContainWord delhi
No webpage contains word delhi

 queryFindPagesWhichContainWord stack
stack_cprogramming, stack_datastructure_wiki, stack_oracle, stacklighting, stackmagazine, stackoverflow

 queryFindPagesWhichContainWord wikipedia
stack_datastructure_wiki

 queryFindPositionsOfWordInAPage magazines stack_datastructure_wiki
Webpage stack_datastructure_wiki does not contain word magazines

 queryFindPagesWhichContainWord allain
stack_cprogramming

 queryFindPagesWhichContainWord allain
stack_cprogramming

 queryFindPagesWhichContainWord C
stack_cprogramming

 queryFindPagesWhichContainWord C++
stack_cprogramming, stackoverflow

 queryFindPagesWhichContainWord jdk
stack_oracle

 queryFindPagesWhichContainWord function
stack_cprogramming, stack_datastructure_wiki, stackoverflow

 queryFindPagesWhichContainWord magazines
stackmagazine
